In [2]:
# 1. Confirm we have a GPU with enough VRAM for 7B in bf16 (~15 GB weights).
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [4]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/afsp-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD 

Cloning into 'Style-Aware-MT'...
remote: Enumerating objects: 402, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 402 (delta 4), reused 7 (delta 4), pack-reused 371 (from 1)
Receiving objects: 100% (402/402), 6.63 MiB | 22.42 MiB/s, done.
Resolving deltas: 100% (215/215), done.
/content/Style-Aware-MT
9b67773
✅


In [ ]:
!pip install -q "transformers==5.12.1" "accelerate==1.14.0" "PyYAML==6.0.3"

In [5]:
!python -m src.infer.run --condition zeroshot --config configs/qwen_smoke.yaml

config.json: 100% 663/663 [00:00<00:00, 3.34MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 78.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 117MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 134MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 49.7MB/s]
Fetching 4 files: 100% 4/4 [03:00<00:00, 45.21s/it] 
Download complete: 100% 15.2G/15.2G [03:01<00:00, 84.1MB/s]                
Loading weights: 100% 339/339 [00:42<00:00,  7.94it/s]
generation_config.json: 100% 243/243 [00:00<00:00, 1.33MB/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Generating 5 translations with Qwen/Qwen2.5-7B-Instruct (zeroshot) ...
  5/5
Wrote outputs/zeroshot_val.jsonl
Usage: {'calls': 5, 'prompt_tokens': 1282, 'completion_tokens': 179, 'cost_usd': 0.0}


In [6]:
# 5. Verify outputs: 5 predictions + non-zero token accounting.
!cat outputs/zeroshot_val_usage.json
print('--- predictions ---')
!wc -l outputs/zeroshot_val.jsonl
import json
with open('outputs/zeroshot_val.jsonl', encoding='utf-8') as f:
    row = json.loads(f.readline())
print('sample prediction:', row['prediction'][:300])

{
  "condition": "zeroshot",
  "model": "Qwen/Qwen2.5-7B-Instruct",
  "calls": 5,
  "prompt_tokens": 1282,
  "completion_tokens": 179,
  "cost_usd": 0.0
}--- predictions ---
5 outputs/zeroshot_val.jsonl
sample prediction: Blessed are the righteous who drink from these rivers, O Thou, the Almighty, the Forgiver, unto whom approacheth none save those who draw nigh by power and might.


## Retrieval / few-shot smoke 

In [7]:
!pip install -q "sentence-transformers==5.5.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 37.7 MB/s eta 0:00:00
✅


In [8]:
!python -m src.retrieval.build_index --config configs/qwen_smoke.yaml

Embedding 10860 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
modules.json: 100% 349/349 [00:00<00:00, 1.73MB/s]
config_sentence_transformers.json: 100% 128/128 [00:00<00:00, 725kB/s]
README.md: 100% 140k/140k [00:00<00:00, 114MB/s]
sentence_xlm-roberta_config.json: 100% 53.0/53.0 [00:00<00:00, 278kB/s]
config.json: 100% 690/690 [00:00<00:00, 4.09MB/s]
model.safetensors: 100% 1.12G/1.12G [00:08<00:00, 133MB/s]
Loading weights: 100% 391/391 [00:00<00:00, 21266.04it/s]
tokenizer_config.json: 100% 1.18k/1.18k [00:00<00:00, 3.08MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 3.94MB/s]
tokenizer.json: 100% 17.1M/17.1M [00:01<00:00, 13.2MB/s]
special_tokens_map.json: 100% 964/964 [00:00<00:00, 4.43MB/s]
config.json: 100% 271/271 [00:00<00:00, 1.51MB/s]
Batches: 100% 340/340 [00:16<00:00, 20.69it/s]
Wrote index to data/knn_index/ : embeddings (10860, 1024), 10860 pairs
✅


In [9]:
# Target-register centroid over train targets -> results/stylometrics_centroid.json.
!python -m src.eval.stylometrics --build-centroid


Target-register centroid  (n=10860)  -> results/stylometrics_centroid.json
---------------------------------------
  lex_density  mean 0.4344   std 0.1101
  ttr          mean 0.8540   std 0.1085
  root_ttr     mean 4.0437   std 1.0426
  marker_rate  mean 0.0327   std 0.0567

✅


In [10]:
# knn_fewshot: cosine top-k retrieval (isolates relevance-based selection over "having examples").
!python -m src.infer.run --condition knn_fewshot --config configs/qwen_smoke.yaml

Retrieving k=4 exemplars for 5 sources (most_similar_last) ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100% 391/391 [00:00<00:00, 6779.46it/s]
config.json: 100% 663/663 [00:00<00:00, 4.19MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 30.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 119MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 123MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 152MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 88.5MB/s]
Fetching 4 files: 100% 4/4 [04:02<00:00, 60.74s/it] 
Download complete: 100% 15.2G/15.2G [04:02<00:00, 62.7MB/s]               
Loading weights: 100% 339/339 [00:49<00:00,  6.83it/s]
generation_config.json: 100% 243/243 [00:00<00:00, 1.58MB/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Generating 5 translations with Qwen/Qwen2.5-7B-Instruct (knn_fewshot) ...
  5/5
Wrote outputs/knn_fewshot_val.jsonl
Usage: {'calls': 5, 'prompt_toke

In [11]:
!python -m src.infer.run --condition afsp_full --config configs/qwen_smoke.yaml

afsp_full: selecting k=4 for 5 sources (most_similar_last) ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100% 391/391 [00:00<00:00, 711.20it/s]
Loading weights: 100% 339/339 [00:46<00:00,  7.32it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Generating 5 translations with Qwen/Qwen2.5-7B-Instruct (afsp_full) ...
  5/5
Wrote outputs/afsp_full_val.jsonl
Usage: {'calls': 5, 'prompt_tokens': 5830, 'completion_tokens': 199, 'cost_usd': 0.0}
✅


In [ ]:
# Verify the few-shot path beyond "no crash": 5 rows each, clean English, no `Source:` leakage.
import json, os
for cond in ("knn_fewshot", "afsp_full"):
    path = f"outputs/{cond}_val.jsonl"
    if not os.path.exists(path):
        print(f"=== {cond}: MISSING ({path}) -- its generation cell did not complete; "
              f"re-run it and read its traceback ===\n")
        continue
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    r = rows[0]
    pred = r["prediction"]
    print(f"=== {cond}: {len(rows)} rows, {sum('error' in x for x in rows)} errors ===")
    print("sample prediction[:250]:", pred[:250])
    print("`Source:` leakage in prediction:", "Source:" in pred)
    print()

=== knn_fewshot: 5 rows, 0 errors ===
sample prediction[:250]: Jewels of the mysteries in the ascent of journeys for him who desireth to draw nigh to God, the Almighty, the Forgiver; blessed indeed are the righteous who drink of these rivers.
`Source:` leakage in prediction: False

=== afsp_full: 5 rows, 0 errors ===
sample prediction[:250]: Blessed are the righteous who drink from these rivers, O my God, for those desirous of drawing nigh unto the Almighty, the Pardoner, through Whose stations of hidden mysteries lie the steps of exalted journeys.
`Source:` leakage in prediction: False

✅
